# Chapter 19 — From Experiment to Production

**Book alignment:** DSPy From First Principles, Chapter 19

**Question this notebook isolates:** Does the ordered promotion gate reject a 0.95 candidate with a hard regression while promoting a clean 0.81 candidate?


In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

from common.fingerprints import fingerprint
from common.promotion import (
    ComparisonDecision,
    ComparisonPolicy,
    DeploymentManifest,
    EvaluationSummary,
    compare,
)

spec = importlib.util.spec_from_file_location(
    "ch17_promo_run", str(EXP_ROOT / "ch17_promotion" / "run.py")
)
promo = importlib.util.module_from_spec(spec)
sys.modules["ch17_promo_run"] = promo
spec.loader.exec_module(promo)

BASELINE_FP = fingerprint({"program_id": "editorial-rewrite-v1", "state": "frozen"})
PROTO, PROTO_OTHER = "protocol-fingerprint-v1", "protocol-fingerprint-v2"
POLICY = ComparisonPolicy(min_cases=2, min_improvement=0.05, expected_baseline_fingerprint=BASELINE_FP)
print("promotion policy: lineage -> protocol -> sufficiency -> regressions -> margin")


## Ordered gates beat blends

The aggregate score is checked last, after lineage, protocol, sufficiency, and both regression gates. Seven synthetic scenarios exercise all three decision classes.


In [ ]:
def summary(program_id, score, hard, valid, cases=3, base_fp=BASELINE_FP, proto=PROTO):
    return EvaluationSummary(program_id, cases, score, hard, valid, base_fp, proto)

active = summary("editorial-rewrite-v1", 0.72, 0, 0)
scenarios = [
    ("clean improvement 0.81", summary("cand-clean", 0.81, 0, 0)),
    ("hard regression 0.95", summary("cand-hard", 0.95, 1, 0)),
    ("validation regression 0.95", summary("cand-valid", 0.95, 0, 1)),
    ("below threshold 0.75", summary("cand-small", 0.75, 0, 0)),
    ("too few cases 0.95", summary("cand-few", 0.95, 0, 0, cases=1)),
    ("protocol mismatch 0.95", summary("cand-proto", 0.95, 0, 0, proto=PROTO_OTHER)),
    ("stale baseline 0.95", summary("cand-stale", 0.95, 0, 0, base_fp="stale-fingerprint")),
]
decisions = [(name, compare(active, cand, POLICY)) for name, cand in scenarios]
for name, rec in decisions:
    print(f"{name:26s} -> {rec.decision.value:12s} ({rec.reason})")
counts = {d.value: sum(1 for _, r in decisions if r.decision.value == d.value) for d in ComparisonDecision}
print("counts:", counts)


In [ ]:
by_name = {name: rec for name, rec in decisions}
assert counts == {"PROMOTE": 1, "REJECT": 2, "INSUFFICIENT": 4}
assert by_name["clean improvement 0.81"].decision == ComparisonDecision.PROMOTE
assert by_name["hard regression 0.95"].decision == ComparisonDecision.REJECT
assert by_name["below threshold 0.75"].decision == ComparisonDecision.INSUFFICIENT
print("five 0.95 candidates did not activate; the clean 0.81 did")


## Promotion is an executable gate

Activation refuses anything but an actual `PROMOTE` record, so the rejected 0.95 hard-regression candidate cannot reach production even though it outscores the baseline.


In [ ]:
prev = DeploymentManifest("deployment-v1", "editorial-rewrite-v1", BASELINE_FP, "eligible")
cand = DeploymentManifest("deployment-v2", "editorial-rewrite-v2", "candidate-fp-v2", "candidate")
promote_rec = by_name["clean improvement 0.81"].to_record()
reject_rec = by_name["hard regression 0.95"].to_record()
kept_prev, active_now, activation = promo.activate_after_promotion(
    previous_manifest=prev, candidate_manifest=cand, promotion_decision=promote_rec
)
blocked, error = False, None
try:
    promo.activate_after_promotion(previous_manifest=prev, candidate_manifest=cand, promotion_decision=reject_rec)
except RuntimeError as exc:
    blocked, error = True, str(exc)
print("activated:", active_now.program_id, "->", active_now.status)
print("rejected activation blocked:", blocked, "|", error)


In [ ]:
assert active_now.status == "active" and active_now.program_id == "editorial-rewrite-v2"
assert blocked is True
print("decision and activation path are coupled: no PROMOTE record, no activation")


## Rollback is rehearsed lineage

Reversal selects the most recent eligible predecessor and writes a tracked rollback record. Candidate, decision, manifest, activation, and rollback each carry a separate fingerprint.


In [ ]:
restored, rollback_rec = promo.rehearse_rollback(history=[kept_prev, active_now], current_active=active_now)
activation_rec = activation.to_record()
rollback_json = rollback_rec.to_record()
fps = [promote_rec["fingerprint"], activation_rec["activation_fingerprint"], rollback_json["rollback_fingerprint"]]
print("restored:", restored.program_id, "->", restored.status, "| reason:", rollback_rec.reason)
print("distinct fingerprints:", len(set(fps)) == 3)


In [ ]:
assert restored.program_id == "editorial-rewrite-v1" and restored.status == "active"
assert rollback_rec.status == "rolled_back"
assert len(set(fps)) == 3
print("rollback restores the eligible predecessor as a recorded event, not an edit")


## What we earned

Independent evaluation, comparison, promotion, activation, and rollback are separate authorities with separate records. An optimizer score can justify comparison; it has no authority to activate a program by itself.

Notebook 20 / Chapter 20 asks whether every mechanism composes into a single loop without any stage certifying its own output.
